# HW5: Evaluate Before You Automate

**COMPSS 211A | Fall 2026 | Student copy**

Researchers use LLMs to label text so they don't have to read all of it. Before you use such labels, you need to check them against texts you labeled yourself. In this homework you do that for your project, with the 50 texts you labeled in HW4. You set two simple baselines, write your Gemini instruction on 20 of the texts, change it once, and then score it on the other 30. Then you decide whether you would use Gemini to label the rest of your project data.

**Due:** Sunday, November 22, 2026 at 11:59 p.m.

Fill in every cell marked "Your turn" and answer the written questions. When a cell asks for a prediction, write your guess before you run anything. Keep the variable names we ask for, because we look for them when grading.

## What you need

- `project_texts.csv` from HW4 Task 6, in the same folder as this notebook. It has one row per text and the columns `id`, `text` and `label`.
- Your team's system instruction and labels from the end of Lab 7.
- Your Gemini API key in the `.env` file in the top folder of the course repository. See "API keys" in `SETUP.md`. Never paste the key into the notebook.
- About 70 calls to Gemini. They take 10 to 15 minutes and cost a few cents at most. Each answer is saved to a file right away, so you only call Gemini once for each text.

If your key doesn't work, ask on Slack or come to office hours before the deadline. Everything you send to Gemini goes to Google, so leave out texts with names or other details that identify a person.

## What you will practice

- Choosing between words, embeddings and labels for a research question.
- Setting up baselines before judging a model.
- Writing a prompt on some texts and scoring it on others.
- Scoring labels with accuracy, a confusion table, and false positives and false negatives.
- Reading the texts where Gemini disagrees with you.

## AI and collaboration policy

Tasks 4 to 6 use Gemini to label your texts. Beyond that, you may use AI to help you understand an error message, as long as you say so in a note in your notebook. Write the code and the written answers yourself.

In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import warnings

import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)  # a harmless warning from one of the libraries

pd.set_option("display.max_colwidth", 300)  # show more of each text

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## 1. Words, embeddings or labels?

| | Words (TF-IDF) | Embeddings | LLM labels |
|---|---|---|---|
| Question | Which words go with a group? | Which texts are about the same thing? | Does each text have the property you define? |
| You need | Groups you already have | Only the texts | A definition, and texts you labeled by hand |
| It misses | Meaning beyond single words | What a text says about its topic | It costs money and can be wrong |

### Your response

Answer in two or three sentences:

1. What is your team's research question?
2. Which of the three methods answers it, and why? If it is labels, which label do you most need to find? That is your *positive* label in Task 6.

> Write your response here, then delete this line.

## 2. Your texts and two baselines

Before you judge Gemini, check how well you could do without it. Two baselines:

- **Most common label:** always guess the label that most of your texts have.
- **Keyword rule:** label a text with your positive label if it contains one of your keywords, and with another label if it doesn't.

**Your turn:**

1. Make `project_majority_accuracy`: the accuracy of always guessing your most common label (hint: `project_texts["label"].value_counts(normalize=True).max()`).
2. Set `POSITIVE_LABEL`, `OTHER_LABEL` and `KEYWORDS`: three to six words that you think mark your positive label.
3. Make the column `keyword_label`, then `keyword_accuracy`: how often `keyword_label` is the same as your label.
4. In a comment, write down the accuracy you expect Gemini to get.

In [ ]:
project_texts = pd.read_csv("project_texts.csv", dtype={"id": str, "label": str})
print(len(project_texts), "texts")  # Check: 50 texts
project_texts["label"].value_counts()

In [ ]:
# Your turn
project_majority_accuracy = ...

POSITIVE_LABEL = "..."  # the label you most need to find
OTHER_LABEL = "..."  # the label the keyword rule gives every other text
KEYWORDS = ["...", "..."]
pattern = r"\b(?:" + "|".join(KEYWORDS) + r")\b"
project_texts["keyword_label"] = ...  # hint: project_texts["text"].str.contains(pattern, case=False) gives True or False
keyword_accuracy = ...

# My prediction for Gemini's accuracy:
#

## 3. Write on 20 texts, score on 30

If you change the prompt until it fits the texts you are looking at, its score on those texts is too high. So you split your 50 texts: you read Gemini's mistakes on 20 texts and change the instruction there, and you score the final instruction once on the other 30. Everyone in your team gets the same split because `random_state` is fixed.

In [ ]:
write_texts = project_texts.sample(n=20, random_state=211)
score_texts = project_texts.drop(write_texts.index)
print(len(write_texts), "texts to write the prompt on;", len(score_texts), "to score it on")  # Check: 20 and 30

## 4. Ask Gemini about the 20 texts

The first three cells are supplied. The first connects to Gemini with your key. The second has two functions: `ask_gemini` sends one text and returns the label and the reason, and `label_texts` calls it for each text in a table and saves each answer to a file right away. If a cell stops partway, run it again: it skips the texts that are already in the file.

**Your turn:** paste your team's system instruction from the end of Lab 7 into `PROJECT_SYSTEM`, and your labels into `PROJECT_LABELS`, as in HW4. Then label the 20 texts and make `first_accuracy`: how often Gemini's label is the same as yours on these 20.

In [ ]:
# Supplied: connect to Gemini. The key is read from your .env file and never printed.
import logging

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv(COURSE_ROOT / ".env")
key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"  # use the replacement model if bCourses announces one

# Hide a warning that the google-genai package prints on every call. It does not mean anything went wrong.
logging.getLogger("google_genai.types").setLevel(logging.ERROR)
print("Connected to Gemini.")

In [ ]:
# Supplied: send one text, and label every text in a table.

# Errors worth trying again, because they go away on their own:
# 429 means too many calls in one minute, and 503 means the server is busy.
# Other errors, like 400 or 404, mean the request itself is wrong. Trying again won't help.
RETRY_CODES = {429, 503}


def ask_gemini(prompt, system, schema, max_tries=4):
    """Send one prompt with this system instruction and schema. Return the label and the reason. Try again after a temporary error."""
    wait = 10  # seconds to wait before the first retry
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    response_mime_type="application/json",
                    response_schema=schema,
                ),
            )
            time.sleep(6)  # wait between calls to stay under the free tier's limit per minute
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in RETRY_CODES or attempt == max_tries:
                raise  # stop and show the error
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")
            time.sleep(wait)
            wait = wait * 2  # wait longer each time: 10, 20, 40 seconds


def label_texts(texts, system, schema, path):
    """Ask Gemini about every text in texts. Save each answer to path right away."""
    # Texts that are already in the file were labeled in an earlier run, so skip them.
    done = set(pd.read_csv(path, dtype={"id": str})["id"]) if path.exists() else set()
    for text in texts.itertuples():
        if text.id in done:
            continue
        reply = ask_gemini(f"Text: {text.text}", system, schema)
        row = pd.DataFrame([{"id": text.id, "label": text.label, **reply, "model": GEMINI_MODEL}])
        # mode="a" adds one line to the end of the file. Write the column names only if the file is new.
        row.to_csv(path, mode="a", header=not path.exists(), index=False)
        done.add(text.id)
        print(len(done), "of", len(texts), "done")
    results = pd.read_csv(path, dtype={"id": str, "label": str, "llm_label": str})
    return results[results["id"].isin(texts["id"])]

In [ ]:
PROJECT_SYSTEM = """..."""  # Your turn: your team's system instruction from the end of Lab 7
PROJECT_LABELS = ["...", "...", "unclear"]  # the same labels as in HW4

assert "..." not in PROJECT_SYSTEM and "..." not in PROJECT_LABELS, "Fill in PROJECT_SYSTEM and PROJECT_LABELS."
PROJECT_SCHEMA = {
    "type": "object",
    "properties": {
        "llm_label": {"type": "string", "enum": [str(label) for label in PROJECT_LABELS]},  # labels must be text, so 1 becomes "1"
        "llm_reason": {"type": "string"},
    },
    "required": ["llm_label", "llm_reason"],
}

In [ ]:
# Supplied: label the 20 texts with your instruction. 20 calls.
first_results = label_texts(write_texts, PROJECT_SYSTEM, PROJECT_SCHEMA, GENERATED_DIR / "hw5_first.csv")
first_results[["id", "label", "llm_label", "llm_reason"]].head()

In [ ]:
# Your turn: first_accuracy, and the texts where Gemini's label differs from yours (with the text, merged from write_texts on id)
first_accuracy = ...
first_disagreements = ...

## 5. Change the instruction once

Read the texts in `first_disagreements` and Gemini's reasons. For each one, ask: is Gemini wrong, am I wrong, or is the definition unclear?

**Your turn:** copy `PROJECT_SYSTEM` into `NEW_SYSTEM` and change one thing: a clearer definition, the line between two close labels, a rule for `unclear`, or an example taken from these 20 texts. Then label the same 20 texts again and make `second_accuracy`.

In [ ]:
# Your turn: one change
NEW_SYSTEM = """..."""
assert "..." not in NEW_SYSTEM, "Write your changed instruction in NEW_SYSTEM."

In [ ]:
# Supplied: label the same 20 texts with the new instruction. 20 calls.
second_results = label_texts(write_texts, NEW_SYSTEM, PROJECT_SCHEMA, GENERATED_DIR / "hw5_second.csv")
second_accuracy = ...  # Your turn

### Your response

Answer in two or three sentences:

1. What did you change, and which disagreement made you change it?
2. Did the change help on the 20 texts? Why is this number too optimistic?

> Write your response here, then delete this line.

## 6. Score once on the other 30

Pick the instruction you trust more and score it once on the 30 texts you have not looked at. Don't change it after this.

**Your turn:**

1. Set `FINAL_SYSTEM` to `PROJECT_SYSTEM` or `NEW_SYSTEM`, and run the cell that labels the 30 texts.
2. Make `final_accuracy`, and the baselines on the same 30 texts: `score_majority_accuracy` (always the most common label in all 50) and `score_keyword_accuracy`.
3. Make `final_confusion` with `pd.crosstab(final_results["label"], final_results["llm_label"])`. Rows are your labels and columns are Gemini's.
4. Make `accuracy_by_label`: Gemini's accuracy for each of your labels (hint: a True/False column, then `groupby("label")`).
5. Make `false_positives` and `false_negatives` for `POSITIVE_LABEL`: texts Gemini gave your positive label that you did not, and texts you gave it that Gemini did not.

In [ ]:
FINAL_SYSTEM = ...  # Your turn: PROJECT_SYSTEM or NEW_SYSTEM

# Supplied: label the 30 texts with the final instruction. 30 calls.
final_results = label_texts(score_texts, FINAL_SYSTEM, PROJECT_SCHEMA, GENERATED_DIR / "hw5_final.csv")
len(final_results)  # Check: 30

In [ ]:
# Your turn
final_accuracy = ...
score_majority_accuracy = ...
score_keyword_accuracy = ...
final_confusion = ...
accuracy_by_label = ...
false_positives = ...
false_negatives = ...

### Your notes on three disagreements in the 30 texts

| id | My label | Gemini | Who is right: me, Gemini, or is the definition unclear? |
| --- | --- | --- | --- |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

## 7. Would you use it?

### Your response

Answer in four or five sentences:

1. On the 30 texts, how did Gemini compare with the two baselines? Which of your labels did it get right, and which did it miss?
2. Which mistake matters more for your research question: false positives or false negatives? Why?
3. Would you use Gemini to label the rest of your project data? If not, what would you change first: the definitions, the prompt, or the method?
4. What would you report about this check in your final project? Name the model, the date, and which texts you used to write the instruction.

> Write your response here, then delete this line.